# Scenario 1 Workbench: Customer Support Resolution Agent

Welcome to the hands-on offline lab for **Scenario 1**. Before running these cells, read [`SCENARIO_GUIDE.md`](SCENARIO_GUIDE.md) for the full architecture walkthrough and 6 practice questions.

### What You Will Observe in This Lab
This notebook runs [`simulation.py`](simulation.py) completely offline (zero API calls) so you can watch how:
1. **Code gates** block order lookups and refunds until customer identity is verified.
2. **Output trimming (`PostToolUse`)** strips ~40 raw database fields down to the 5 fields the agent needs.
3. **Explicit escalation** hands off immediately when a customer asks for a human or exceeds the `$500` refund cap.
4. **Structured handoffs & errors** return machine-readable `isError` / `isRetryable` flags and keep verified Case Facts intact.

**Memory Hook:** *"Money in code, humans on request, facts in a block."*

In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_1_customer_support.simulation import *

print(' Scenario Workbench Initialized: Scenario 1: Customer Support Resolution Agent')
print(' Zero live API calls, 100% offline determinism')


### Step 2: Run the Happy Path (Verify Identity, Trim Order Fields, Issue Refund)
Watch how the pipeline authenticates `alice@example.com`, strips internal warehouse/audit fields from `lookup_order`, and approves an eligible refund within the 30-day window.

In [ ]:
# Execute Happy Path: Authentication + Return Lookup with Trimming
pipeline = CustomerSupportPipeline()

# 1. Auth
auth_res = pipeline.process_customer_turn("My email is alice@example.com")
print("Auth action:", auth_res["action_taken"])
assert pipeline.case_facts.is_authenticated is True

# 2. Lookup order
order_res = pipeline.process_customer_turn("Look up ord_9901")
print("Order action:", order_res["action_taken"])
print("Trimmed Payload:", order_res["order_data"])
assert "_internal_warehouse_id" not in order_res["order_data"]

# 3. Process refund
refund_res = pipeline.process_customer_turn("Please issue the refund.")
print("Refund action:", refund_res["action_taken"])
assert refund_res["action_taken"] == "REFUND_SUCCESS"



### Step 3: Test Safety Gates & Immediate Human Escalation
Now see what happens when an unauthenticated user asks to check an order (blocked in code) or explicitly demands to speak with a human (escalated immediately with zero extra questions).

In [ ]:
# Execute Edge Cases: Unauthenticated Refund Attempt & Immediate Escalation
pipeline2 = CustomerSupportPipeline()

# Edge Case 1: Unauthenticated lookup blocked
res_blocked = pipeline2.process_customer_turn("Please check ord_9901")
print("Blocked action:", res_blocked["action_taken"])
assert res_blocked["action_taken"] == "GATE_BLOCKED_UNAUTHENTICATED"

# Edge Case 2: Immediate human escalation
res_esc = pipeline2.process_customer_turn("I want to speak to a human right now.")
print("Escalation action:", res_esc["action_taken"])
assert res_esc["action_taken"] == "ESCALATED_TO_HUMAN"
assert pipeline2.escalated is True



### Step 4: Verify Refund Thresholds, Structured Errors & Handoff History
Finally, test direct tool-gate error responses (`isError: true`, `isRetryable: false`) and confirm that a `$750` refund (above the `$500` policy limit) automatically escalates with `actions_taken` preserved for the human agent.

In [ ]:
pipe = CustomerSupportPipeline()
blocked = pipe.execute_tool("process_refund", order_id="ord_9901")
assert blocked["isError"] is True and blocked["isRetryable"] is False
pipe.process_customer_turn("My email is bob@example.com")
res = pipe.process_customer_turn("Refund ord_9903")  # $750 > $500 limit
assert res["action_taken"] == "ESCALATED_TO_HUMAN"
assert pipe.escalation_data.escalation_type == "THRESHOLD"
print("Handoff actions:", pipe.escalation_data.actions_taken)


### Lab Complete
All offline checks passed. For full unit tests, run `uv run pytest scenarios/scenario_1_customer_support/test_scenario_1_simulation.py` in WSL Ubuntu-26.04.